### Dynamic Hedging with Bond Portfolio

We want to engage in a 3-year investment with a return goal of $100,000

Assume interest rate at t = 0 is 12%

Assume there are only 2 products: 
- Bond A: 5-year coupon bond with annual coupon C = $10 and face value F = $100
- Bond B: 1-year zero-coupon bond with face value F = $100

In [2]:
import numpy as np

In [22]:
C = 10
F = 100
r = 0.12

# Price of the bonds
def price_coupon_bond(CFs, r):
    return round(np.sum([CF * np.exp(-(t +1) * r) for t,CF in enumerate(CFs)]), 3)

def duration_coupon_bond(CFs, r):
    d = 1/price_coupon_bond(CFs, r)
    d *= np.sum([(t+1) * CF * np.exp(-(t +1) * r) for t,CF in enumerate(CFs)])
    return round(d, 3)

def price_zerocoupon_bond(F, t, r):
    return round(F * np.exp(-t * r), 3)


Pa_t0 = price_coupon_bond([C, C, C, C, F+C], r)
Pb_t0 = price_zerocoupon_bond(F, 1, r)

print(f"The price of bond A at t = 0 is ${Pa_t0}")
print(f"The price of bond B at t = 0 is ${Pb_t0}")

The price of bond A at t = 0 is $90.269
The price of bond B at t = 0 is $88.692


In [30]:
# Initial investment, given r = 0.12, we invest present day value of $100,000 to be collected in 3 years
P = 100_000 * np.exp(-3 * r)

print(f"The initial investment is given by {round(P, 2)}")

The initial investment is given by 69767.63


In [23]:
# Find Duration numerically
h = 1e-4
Da_t0_num = -1/Pa_t0 * (price_coupon_bond([C, C, C, C, F+C], r + h) - price_coupon_bond([C, C, C, C, F+C], r - h)) / (2 * h)
Db_t0_num = -1/Pa_t0 * (price_zerocoupon_bond(F, 1, r + h) - price_zerocoupon_bond(F, 1, r - h)) / (2 * h)

Da_t0 = duration_coupon_bond([C, C, C, C, F+C], r)
Db_t0 = 1

print(f"The Duration of bond A at t = 0 is {Da_t0} [{round(Da_t0_num, 3)}]")
print(f"The Duration of bond B at t = 0 is {Db_t0} [{round(Db_t0_num, 3)}]")

The Duration of bond A at t = 0 is 4.122 [4.154]
The Duration of bond B at t = 0 is 1 [0.997]


Note that the portfolio can be build using the following structure

$$\begin{align}
    P &= a \cdot P_A(y(0)) + b \cdot P_B(y(0)) \\
    D &= \omega_A D_A(y(0)) + \omega_B D_B(y(0)) \\
    \omega_A &= a \frac{P_A(y(0))}{P} \\
    \omega_B &= b \frac{P_B(y(0))}{P} 
\end{align}$$

By solving a systme of linear equation we find

$$
\begin{pmatrix}
1 \\
D
\end{pmatrix}
=
\begin{pmatrix}
1 & 1 \\
D_A & D_B
\end{pmatrix}
\begin{pmatrix}
w_A \\
w_B
\end{pmatrix}
$$

$$
\begin{pmatrix}
w_A \\
w_B
\end{pmatrix}
=
\begin{pmatrix}
1 & 1 \\
D_A & D_B
\end{pmatrix}^{-1}
\begin{pmatrix}
1 \\
D
\end{pmatrix}

$$

In [32]:
# We need to build a bond portfolio with Duration D = 3 to make it stable to changes in interest rate 
# (plan is to invest for 3 years)
# Solve to find wA and wB for building portfolio with Duration = 3
D = 3

y = np.array([1, D])
A = np.array([[1, 1],
              [Da_t0, Db_t0]])
w = np.linalg.solve(A, y)

print(f"The weights to achieve D = 3 are: wA = {round(w[0], 4)}, wB = {round(w[1], 4)}")

a = w[0] * P / Pa_t0
b = w[1] * P / Pb_t0

print(f"The weights to build a portfolio with D = 3 are: a = {round(a, 4)}, b = {round(b, 4)}")

The weights to achieve D = 3 are: wA = 0.6406, wB = 0.3594
The weights to build a portfolio with D = 3 are: a = 495.1223, b = 282.7024


To summarize up to now, we have a goal of $100,000 in 3 years

We start by investing $69,737.70 in a portfolio of bonds
- We buy 495.1223 bonds of type A spending ~$44,694.19
- We buy 282.7024 bonds of type B spending ~$25,073.44

### How will our new portfolio (designed to be immune to interest rate changes) behave?

#### SCENARIO 1: After one year the interest rate increases to 14%
At t = 1:
- First coupon is due for a total of $10 x 495.122 = $4,951.22
- Zero-coupon bond matures for a total of $100 x 282.7024 = $28,270.24
- Value of bond A drops due to interest rate increasing to $


#### SCENARIO 2: After one year the interest rate drops to 9%